# **III. Análisis de identificadores: pacientes y hospitales**

## Objetivo
Este notebook busca evaluar el rol de las variables identificadoras (CIP_ENCRIPTADO/ID_BENEFICIARIO y COD_HOSPITAL) dentro del modelado predictivo para determinar su viabilidad como variables predictoras en los modelos de aprendizaje automático.

## Proceso
1. **Análisis de pacientes repetidos**: Cuantifica la frecuencia de episodios oncológicos por paciente único (CIP_ENCRIPTADO) durante 2019 y 2024
2. **Análisis de distribución de hospitales**: Analiza la distribución de episodios oncológicos por hospital (COD_HOSPITAL) para evaluar su cardinalidad e impacto

#### Identificador de pacientes repetidos: CIP_ENCRIPTADO o ID_BENEFICIARIO

1. **CID_ENCRIPTADO (o ID_BENEFICIARIO)**

- **Tipo de variable:** Demográfica (identificador del paciente).
- **Descripción:** Código identificador único (encriptado) de cada paciente que ingresa al sistema hospitalario.
- **Veredicto:** Descartar (como variable predictora en los modelos). Porque un número de identificación no tiene ninguna relación clínica ni operativa con la criticidad del cáncer o consumo de recursos. Además, su cardinalidad es gigantesca (decenas de miles de valores únicos por año) y al intentar aplicar One-Hot Encoding, se crearían decenas de miles de columnas nuevas (lo que puede introducir la maldición de la dimensionalidad en los modelos). Finalmente, en la propuesta se estableció que cada evento se caracterizará de forma independiente, considerando múltiples episodios de un mismo paciente. Por lo tanto, el hecho de que un paciente se repita (por ejemplo, 6.798 veces en 2019) es un comportamiento esperado de la enfermedad crónica, pero cada ingreso debe evaluarse por las variables de ese momento (diagnóstico, tipo de ingreso, entre otras), no por su ID.

In [3]:
import pandas as pd # Para manipulación de datos
import os # Para manejo de rutas y archivos
import warnings # Para suprimir warning

# Suprimir warnings de DtypeWarning
warnings.simplefilter(action='ignore', category=pd.errors.DtypeWarning)

# ============================================================================
# CONFIGURACIÓN INICIAL: Rutas y archivos
# ============================================================================
# Rutas de entrada y salida
carpeta = "../../Datos/Datos oncológicos (sin procesar)"
ruta_resultados = "../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos"
os.makedirs(ruta_resultados, exist_ok=True) # Crear carpeta de resultados si no existe
# Lista de archivos oncológicos a procesar (2019-2024)
archivos = [
    "GRD_ONCOLOGIA_2019.csv",
    "GRD_ONCOLOGIA_2020.csv",
    "GRD_ONCOLOGIA_2021.csv",
    "GRD_ONCOLOGIA_2022.csv",
    "GRD_ONCOLOGIA_2023.csv",
    "GRD_ONCOLOGIA_2024.csv"
]
# Mapeo de columnas para estandarizar nombres inconsistentes
# Nota: "ï»¿COD_HOSPITAL" es una corrupción de encoding (BOM UTF-8 no removido)
mapa_columnas = {
    "ï»¿COD_HOSPITAL": "COD_HOSPITAL",
    "ID_BENEFICIARIO": "CIP_ENCRIPTADO"
}

In [4]:
# ============================================================================
# SECCIÓN 1: ANÁLISIS DE PACIENTES REPETIDOS (CIP_ENCRIPTADO)
# ============================================================================
# Variable para almacenar el top 5 de pacientes repetidos por año
# Será utilizado para análisis comparativo temporal
top5_repetidos_por_año = {}

# Iterar sobre cada año disponible
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    año = archivo[-8:-4]  # Extrae el año del nombre del archivo
    
    # Cargar CSV con low_memory=False para evitar advertencias de tipos mixtos
    df = pd.read_csv(ruta, low_memory=False).rename(columns=mapa_columnas)
    
    # Procesar solo si la columna CIP_ENCRIPTADO existe
    if "CIP_ENCRIPTADO" in df.columns:
        # Convertir a string para evitar comparaciones numéricas problemáticas
        df["CIP_ENCRIPTADO"] = df["CIP_ENCRIPTADO"].astype(str)
        
        # Filtrar registros con información faltante o codificada como desconocida
        df = df[~df["CIP_ENCRIPTADO"].isin(["SIN INFORMACION", "DESCONOCIDO"])]
        
        # Contar la frecuencia de cada CIP (cuántos episodios por paciente)
        conteo = df["CIP_ENCRIPTADO"].value_counts()
        
        # Filtrar solo pacientes con múltiples episodios (count > 1)
        repetidos = conteo[conteo > 1]
        
        # Crear un DataFrame con ranking ordenado de repeticiones
        repetidos_df = repetidos.reset_index()
        repetidos_df.columns = ["CIP_ENCRIPTADO", "count"]
        
        # Guardar ranking completo en CSV para auditoría
        ruta_salida = f"{ruta_resultados}/repetidos_{año}.csv"
        repetidos_df.to_csv(ruta_salida, index=False)
        
        # Almacenar top 5 para visualización posterior
        top5_repetidos_por_año[año] = repetidos_df.head(5)
        
        # Generar reporte de progreso
        print(f"✓ {año}: {len(repetidos_df)} pacientes con múltiples episodios. "
              f"Max: {repetidos.max()} episodios. Guardado en {ruta_salida}")

✓ 2019: 6798 pacientes con múltiples episodios. Max: 50 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2019.csv
✓ 2020: 4368 pacientes con múltiples episodios. Max: 17 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2020.csv
✓ 2021: 5377 pacientes con múltiples episodios. Max: 16 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2021.csv
✓ 2022: 6245 pacientes con múltiples episodios. Max: 15 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2022.csv
✓ 2023: 7766 pacientes con múltiples episodios. Max: 20 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2023.csv
✓ 2024: 8413 pacientes con múltiples episodios. Max: 22 episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Pacientes repetidos/repetidos_2024.csv


In [19]:
# Pacientes oncológicos repetidos entre 2019 y 2021
print("="*70)
print("TOP 5 PACIENTES CON MÁS EPISODIOS ONCOLÓGICOS (2019-2021)")
print("="*70)

for año in ["2019", "2020", "2021"]:
    print(f"\n- Año {año} (Máximo de episodios en {año}: {top5_repetidos_por_año[año]['count'].iloc[0]}):")
    print(top5_repetidos_por_año[año].to_string(index=False))

TOP 5 PACIENTES CON MÁS EPISODIOS ONCOLÓGICOS (2019-2021)

- Año 2019 (Máximo de episodios en 2019: 50):
  CIP_ENCRIPTADO  count
SIN INFORMACIÃN     50
          539578     20
         1255739     20
          941440     19
          279295     18

- Año 2020 (Máximo de episodios en 2020: 17):
CIP_ENCRIPTADO  count
       1032685     17
        357009     14
        594163     13
        565769     13
        117997     12

- Año 2021 (Máximo de episodios en 2021: 16):
CIP_ENCRIPTADO  count
    72032169.0     16
    74801739.0     13
    78474733.0     12
    96306789.0     11
    80250037.0     11


In [21]:
# Pacientes oncológicos repetidos entre 2022 y 2024
print("="*70)
print("TOP 5 PACIENTES CON MÁS EPISODIOS ONCOLÓGICOS (2022-2024)")
print("="*70)

for año in ["2022", "2023", "2024"]:
    print(f"\n- Año {año} (Máximo de episodios en {año}: {top5_repetidos_por_año[año]['count'].iloc[0]}):")
    print(top5_repetidos_por_año[año].to_string(index=False))

TOP 5 PACIENTES CON MÁS EPISODIOS ONCOLÓGICOS (2022-2024)

- Año 2022 (Máximo de episodios en 2022: 15):
CIP_ENCRIPTADO  count
      73769834     15
      72032169     15
      91567673     14
      95033123     13
      94815142     12

- Año 2023 (Máximo de episodios en 2023: 20):
CIP_ENCRIPTADO  count
      73542175     20
      97107693     16
      98086978     15
      77364498     15
      76674648     14

- Año 2024 (Máximo de episodios en 2024: 22):
CIP_ENCRIPTADO  count
      79541137     22
      94810737     18
     100529456     18
      97789159     18
      83025570     16


In [22]:
# Generar análisis de tendencias temporal
print("\n" + "="*70)
print("ANÁLISIS DE TENDENCIAS TEMPORALES (Mayor repetición por año)")
print("="*70)
for año in ["2019", "2020", "2021", "2022", "2023", "2024"]:
    max_rep = top5_repetidos_por_año[año]['count'].iloc[0]
    print(f"  {año}: {max_rep:3d} episodios (paciente con mayor repetición)")


ANÁLISIS DE TENDENCIAS TEMPORALES (Mayor repetición por año)
  2019:  50 episodios (paciente con mayor repetición)
  2020:  17 episodios (paciente con mayor repetición)
  2021:  16 episodios (paciente con mayor repetición)
  2022:  15 episodios (paciente con mayor repetición)
  2023:  20 episodios (paciente con mayor repetición)
  2024:  22 episodios (paciente con mayor repetición)


#### Distribución de hospitales en los episodios oncológicos (COD_HOSPITAL)

2. **COD_HOSPITAL**
- **Tipo de variable:** Hospitalaria.
- **Descripción:** Código numérico que identifica al establecimiento de salud específico donde ocurrió el episodio y el egreso del paciente.
- **Veredicto:** Descartar (como variable predictora en los modelos), ya que tener entre 65 y 72 categorías distintas añade una dimensionalidad muy alta al aplicar One-Hot Encoding. Además, esta variable está anidada jerárquicamente dentro de SERVICIO_SALUD (que se analizará en el siguiente notebook y tiene 31 macro categorías). Mantener ambas generaría colinealidad (los modelos se confundirían al tener dos variables que explican el mismo fenómeno geográfico y operativo). Además, el propósito central de la propuesta es apoyar la planificación sanitaria y la gestión de recursos a nivel macro (red), dirigidos a instituciones como FONASA y MINSAL, y no a decisiones clínicas individuales. 

In [23]:
# ============================================================================
# SECCIÓN 2: ANÁLISIS DE DISTRIBUCIÓN DE HOSPITALES (COD_HOSPITAL)
# ============================================================================

# Crear carpeta de salida para resultados de hospitales
ruta_resultados_hospitales = "../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales"
os.makedirs(ruta_resultados_hospitales, exist_ok=True)

# Diccionario para almacenar el top 5 de hospitales por año
# Será utilizado para análisis de concentración de episodios
top5_hospitales_por_año = {}

# Iterar sobre cada año disponible
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    año = archivo[-8:-4]  # Extrae el año del nombre del archivo
    
    # Cargar y estandarizar nombres de columnas
    df = pd.read_csv(ruta, low_memory=False).rename(columns=mapa_columnas)
    
    # Procesar solo si la columna COD_HOSPITAL existe
    if "COD_HOSPITAL" in df.columns:
        # Convertir a string para mantener consistencia (evita interpretación numérica)
        df["COD_HOSPITAL"] = df["COD_HOSPITAL"].astype(str)
        
        # Contar la frecuencia de episodios por hospital
        conteo_hosp = df["COD_HOSPITAL"].value_counts()
        
        # Crear DataFrame con ranking ordenado de volumen de episodios
        hospitales_df = conteo_hosp.reset_index()
        hospitales_df.columns = ["COD_HOSPITAL", "count"]
        
        # Guardar ranking completo en CSV para auditoría
        ruta_salida_hosp = f"{ruta_resultados_hospitales}/hospitales_{año}.csv"
        hospitales_df.to_csv(ruta_salida_hosp, index=False)
        
        # Almacenar top 5 para visualización posterior
        top5_hospitales_por_año[año] = hospitales_df.head(5)
        
        # Calcular métricas de concentración
        num_hospitales = hospitales_df['COD_HOSPITAL'].nunique()
        top5_volume = hospitales_df.head(5)['count'].sum()
        total_volume = hospitales_df['count'].sum()
        concentracion = (top5_volume / total_volume) * 100
        
        # Generar reporte de progreso
        print(f"✓ {año}: {num_hospitales} hospitales distintos. "
              f"Top 5 concentra {concentracion:.1f}% de episodios. Guardado en {ruta_salida_hosp}")

✓ 2019: 65 hospitales distintos. Top 5 concentra 26.0% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2019.csv
✓ 2020: 65 hospitales distintos. Top 5 concentra 25.6% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2020.csv
✓ 2021: 65 hospitales distintos. Top 5 concentra 27.5% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2021.csv
✓ 2022: 65 hospitales distintos. Top 5 concentra 25.9% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2022.csv
✓ 2023: 68 hospitales distintos. Top 5 concentra 25.7% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/hospitales_2023.csv
✓ 2024: 72 hospitales distintos. Top 5 concentra 24.8% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Distribución de hospitales/

In [ ]:
# Hospitales con más episodios oncológicos entre 2019 y 2021
print("\n" + "="*70)
print("TOP 5 HOSPITALES CON MÁS CANTIDAD DE EPISODIOS ONCOLÓGICOS (2019-2021)")
print("="*70)

for año in ["2019", "2020", "2021"]:
    max_hosp = top5_hospitales_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Hospital con mayor cantidad de episodios: {max_hosp} episodios):")
    print(top5_hospitales_por_año[año].to_string(index=False))


TOP 5 HOSPITALES CON MÁS CANTIDAD DE EPISODIOS ONCOLÓGICOS (2019-2021)

Año 2019 (Hospital con mayor cantidad de episodios: 2889 episodios):
COD_HOSPITAL  count
      114101   2889
      110100   2672
      118100   2561
      106100   2036
      122100   1883

Año 2020 (Hospital con mayor cantidad de episodios: 2155 episodios):
COD_HOSPITAL  count
      118100   2155
      114101   2035
      110100   1689
      122100   1644
      116105   1522

Año 2021 (Hospital con mayor cantidad de episodios: 2775 episodios):
COD_HOSPITAL  count
      118100   2775
      114101   2468
      110100   1984
      122100   1844
      116105   1594


In [29]:
# Hospitales con más episodios oncológicos entre 2022 y 2024
print("\n" + "="*70)
print("TOP 5 HOSPITALES CON MÁS CANTIDAD DE EPISODIOS ONCOLÓGICOS (2022-2024)")
print("="*70)

for año in ["2022", "2023", "2024"]:
    max_hosp = top5_hospitales_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Hospital con mayor cantidad de episodios: {max_hosp} episodios):")
    print(top5_hospitales_por_año[año].to_string(index=False))


TOP 5 HOSPITALES CON MÁS CANTIDAD DE EPISODIOS ONCOLÓGICOS (2022-2024)

Año 2022 (Hospital con mayor cantidad de episodios: 2967 episodios):
COD_HOSPITAL  count
      118100   2967
      110100   2449
      114101   2438
      122100   1944
      116105   1812

Año 2023 (Hospital con mayor cantidad de episodios: 3185 episodios):
COD_HOSPITAL  count
      118100   3185
      114101   2861
      110100   2628
      111100   2314
      122100   2306

Año 2024 (Hospital con mayor cantidad de episodios: 3095 episodios):
COD_HOSPITAL  count
      118100   3095
      114101   2890
      111100   2624
      110100   2559
      122100   2508


In [ ]:
# Genero un análisis de tendencias temporal para hospitales
for año in ["2019", "2020", "2021", "2022", "2023", "2024"]:
    # Recargar datos para obtener totales
    archivo_año = [f for f in archivos if año in f][0]
    ruta_archivo = os.path.join(carpeta, archivo_año)
    df_temp = pd.read_csv(ruta_archivo, low_memory=False).rename(columns=mapa_columnas) # Cargar y estandarizar columnas
    # Contar episodios por hospital
    conteo_temp = df_temp["COD_HOSPITAL"].value_counts()
    hosp_temp_df = conteo_temp.reset_index() # Crear DataFrame con ranking de hospitales
    hosp_temp_df.columns = ["COD_HOSPITAL", "count"] # Calcular métricas de concentración
    total_hosp = len(hosp_temp_df) # Número total de hospitales distintos
    top5_eps = hosp_temp_df.head(5)['count'].sum() # Suma de episodios en los top 5 hospitales
    total_eps = hosp_temp_df['count'].sum() # Suma total de episodios en el año
    conc_pct = (top5_eps / total_eps) * 100 # Porcentaje de concentración en top 5
    
    print(f"  {año}: {total_hosp:2d} hospitales | Concentración (top 5): {conc_pct:5.1f}%")

  2019: 65 hospitales | Concentración (top 5):  26.0%
  2020: 65 hospitales | Concentración (top 5):  25.6%
  2021: 65 hospitales | Concentración (top 5):  27.5%
  2022: 65 hospitales | Concentración (top 5):  25.9%
  2023: 68 hospitales | Concentración (top 5):  25.7%
  2024: 72 hospitales | Concentración (top 5):  24.8%


#### Distribución de médicos (a través de sus identificadores)

3. **MEDICOINTERV1_ENCRIPTADO**
- **Tipo de variable:** Hospitalaria (identificador).
- **Descripción:** Código encriptado que identifica de forma única al médico responsable de realizar la primera intervención o procedimiento quirúrgico al paciente durante su estadía.
- **Veredicto:** Descartar completamente al poder provocar fuga de datos, ya que la asignación de un médico de intervención ocurre después del ingreso del paciente. Además, tiene una cardinalidad enorme (desde 2.507 médicos en 2019 hasta 3.277 en 2024), lo cual incrementaría la dimensionalidad enormemente al aplicar One-Hot Encoding, creando 3.000 columnas altamente dispersas (llenas de ceros), lo que podría destruir la capacidad de generalizar de los modelos basados en árboles. Además, la propuesta establece el objetivo de apoyar la planificación sanitaria a nivel red y el código de un médico individual no aporta ningún conocimiento epidemiológico o clínico.

In [11]:
# ============================================================================
# SECCIÓN 3: ANÁLISIS DE MÉDICOS INTERVINIENTES (MEDICOINTERV1_ENCRIPTADO)
# ============================================================================

# Definir carpeta de resultados para médicos intervinientes
ruta_resultados_med_interv = "../../Resultados/Resultados (etapa 1 y 2)/Médicos de intervención"
os.makedirs(ruta_resultados_med_interv, exist_ok=True)  # Crear carpeta si no existe

# Diccionario para guardar el top 5 médicos más frecuentes por año
top5_med_interv_por_año = {}

# Iterar sobre cada archivo de la lista de años
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)  # Construir ruta completa del archivo
    año = archivo[-8:-4]  # Extraer año desde el nombre del archivo
    
    # Leer archivo CSV con low_memory=False para evitar warnings de tipos mezclados
    df = pd.read_csv(ruta, low_memory=False).rename(columns=mapa_columnas)
    
    # Verificar que la columna de médicos exista en el dataset
    if "MEDICOINTERV1_ENCRIPTADO" in df.columns:
        df["MEDICOINTERV1_ENCRIPTADO"] = df["MEDICOINTERV1_ENCRIPTADO"].astype(str)  # Forzar tipo string
        
        # Eliminar valores inválidos o sin información
        df = df[~df["MEDICOINTERV1_ENCRIPTADO"].isin(["SIN INFORMACION", "DESCONOCIDO", "nan", "None"])]
        
        # Contar frecuencia de aparición de cada médico
        conteo_med = df["MEDICOINTERV1_ENCRIPTADO"].value_counts()
        
        # Convertir a DataFrame con columnas claras
        med_df = conteo_med.reset_index()
        med_df.columns = ["MEDICOINTERV1_ENCRIPTADO", "count"]
        
        # Guardar distribución completa en CSV por año
        ruta_salida = f"{ruta_resultados_med_interv}/medicos_intervencion_{año}.csv"
        med_df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
        
        # Guardar top 5 médicos más frecuentes en memoria para análisis posterior
        top5_med_interv_por_año[año] = med_df.head(5)
        
        # Calcular métricas de concentración
        num_med = med_df["MEDICOINTERV1_ENCRIPTADO"].nunique()  # Número de médicos distintos
        top5_volume = med_df.head(5)["count"].sum()  # Episodios acumulados en top 5
        total_volume = med_df["count"].sum()  # Total de episodios
        concentracion = (top5_volume / total_volume) * 100  # Porcentaje de concentración
        
        # Imprimir resumen en consola
        print(f"✓ {año}: {num_med} médicos intervinientes distintos. "
              f"Top 5 concentra {concentracion:.1f}% de episodios. Guardado en {ruta_salida}")

✓ 2019: 2507 médicos intervinientes distintos. Top 5 concentra 9.6% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Médicos de intervención/medicos_intervencion_2019.csv
✓ 2020: 2601 médicos intervinientes distintos. Top 5 concentra 8.4% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Médicos de intervención/medicos_intervencion_2020.csv
✓ 2021: 2612 médicos intervinientes distintos. Top 5 concentra 8.8% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Médicos de intervención/medicos_intervencion_2021.csv
✓ 2022: 2808 médicos intervinientes distintos. Top 5 concentra 9.5% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Médicos de intervención/medicos_intervencion_2022.csv
✓ 2023: 3037 médicos intervinientes distintos. Top 5 concentra 7.4% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Médicos de intervención/medicos_intervencion_2023.csv
✓ 2024: 3277 médicos intervinientes distintos. Top

In [13]:
for año in ["2019", "2020", "2021"]: # Médicos intervinientes más frecuentes entre 2019 y 2021
    max_hosp = top5_med_interv_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Médico con mayor cantidad de intervenciones: {max_hosp} intervenciones):")
    print(top5_med_interv_por_año[año].to_string(index=False))


Año 2019 (Médico con mayor cantidad de intervenciones: 1599 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
                 11769.0   1599
                    54.0    461
                  9016.0    243
                  2587.0    151
                 11290.0    148

Año 2020 (Médico con mayor cantidad de intervenciones: 947 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
                 11780.0    947
                 11769.0    508
                  8312.0    132
                  3331.0    120
                  6764.0    112

Año 2021 (Médico con mayor cantidad de intervenciones: 1600 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
              98438434.0   1600
              92021088.0    238
              67316012.0    141
              82394844.0    134
              74956461.0    130


In [14]:
for año in ["2022", "2023", "2024"]: # Médicos intervinientes más frecuentes entre 2022 y 2024  
    max_hosp = top5_med_interv_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Médico con mayor cantidad de intervenciones: {max_hosp} intervenciones):")
    print(top5_med_interv_por_año[año].to_string(index=False))


Año 2022 (Médico con mayor cantidad de intervenciones: 1774 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
              98438434.0   1774
              73663126.0    471
              74956461.0    225
              73199545.0    219
              92021088.0    194

Año 2023 (Médico con mayor cantidad de intervenciones: 1308 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
              98438434.0   1308
              73663126.0    594
              73199545.0    304
              74956461.0    191
              80847393.0    180

Año 2024 (Médico con mayor cantidad de intervenciones: 1272 intervenciones):
MEDICOINTERV1_ENCRIPTADO  count
              98438434.0   1272
              73663126.0    550
              77789698.0    390
              74956461.0    255
              73199545.0    202


3. **MEDICOALTA_ENCRIPTADO**
- **Tipo de variable:** Hospitalaria (identificador).
- **Descripción:** Código encriptado que identifica al médico tratante que firma formalmente el egreso (alta, traslado o defunción) del paciente al terminar su episodio clínico.
- **Veredicto:** Descartar completamente, ya que puede provocar fuga de datos extrema, al representar literalmente el final de la hospitalización. Y su cardinalidad es mayor a la del médico intervencionista, superando los 6.400 valores únicos en 2024. La inclusión de esta variable colapsaría la memoria y rendimiento de los modelos por el exceso de columnas generadas mediante One-Hot Encoding.

In [15]:
# ============================================================================
# SECCIÓN 4: ANÁLISIS DE MÉDICOS DE ALTA (MEDICOALTA_ENCRIPTADO)
# ============================================================================

# Definir carpeta de resultados para médicos de alta
ruta_resultados_med_alta = "../../Resultados/Resultados (etapa 1 y 2)/Médicos de alta"
os.makedirs(ruta_resultados_med_alta, exist_ok=True)  # Crear carpeta si no existe

# Diccionario para guardar el top 5 médicos de alta más frecuentes por año
top5_med_alta_por_año = {}

# Iterar sobre cada archivo de la lista de años
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)  # Construir ruta completa del archivo
    año = archivo[-8:-4]  # Extraer año desde el nombre del archivo
    
    # Leer archivo CSV con low_memory=False para evitar warnings de tipos mezclados
    df = pd.read_csv(ruta, low_memory=False).rename(columns=mapa_columnas)
    
    # Verificar que la columna de médicos de alta exista en el dataset
    if "MEDICOALTA_ENCRIPTADO" in df.columns:
        df["MEDICOALTA_ENCRIPTADO"] = df["MEDICOALTA_ENCRIPTADO"].astype(str)  # Forzar tipo string
        
        # Eliminar valores inválidos o sin información
        df = df[~df["MEDICOALTA_ENCRIPTADO"].isin(["SIN INFORMACION", "DESCONOCIDO", "nan", "None"])]
        
        # Contar frecuencia de aparición de cada médico de alta
        conteo_med = df["MEDICOALTA_ENCRIPTADO"].value_counts()
        
        # Convertir a DataFrame con columnas claras
        med_df = conteo_med.reset_index()
        med_df.columns = ["MEDICOALTA_ENCRIPTADO", "count"]
        
        # Guardar distribución completa en CSV por año
        ruta_salida = f"{ruta_resultados_med_alta}/medicos_alta_{año}.csv"
        med_df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
        
        # Guardar top 5 médicos de alta más frecuentes en memoria para análisis posterior
        top5_med_alta_por_año[año] = med_df.head(5)
        
        # Calcular métricas de concentración
        num_med = med_df["MEDICOALTA_ENCRIPTADO"].nunique()  # Número de médicos de alta distintos
        top5_volume = med_df.head(5)["count"].sum()  # Episodios acumulados en top 5
        total_volume = med_df["count"].sum()  # Total de episodios
        concentracion = (top5_volume / total_volume) * 100  # Porcentaje de concentración
        
        # Imprimir resumen en consola
        print(f"✓ {año}: {num_med} médicos de alta distintos. "
              f"Top 5 concentra {concentracion:.1f}% de episodios. Guardado en {ruta_salida}")

✓ 2019: 4898 médicos de alta distintos. Top 5 concentra 18.2% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Médicos de alta/medicos_alta_2019.csv
✓ 2020: 5109 médicos de alta distintos. Top 5 concentra 14.2% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Médicos de alta/medicos_alta_2020.csv
✓ 2021: 5139 médicos de alta distintos. Top 5 concentra 13.5% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Médicos de alta/medicos_alta_2021.csv
✓ 2022: 5356 médicos de alta distintos. Top 5 concentra 14.8% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Médicos de alta/medicos_alta_2022.csv
✓ 2023: 5964 médicos de alta distintos. Top 5 concentra 12.9% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Médicos de alta/medicos_alta_2023.csv
✓ 2024: 6465 médicos de alta distintos. Top 5 concentra 11.6% de episodios. Guardado en ../../Resultados/Resultados (etapa 1 y 2)/Médicos de alta/medicos_alta_2

In [16]:
for año in ["2019", "2020", "2021"]: # Médicos de alta más frecuentes entre 2019 y 2021
    max_hosp = top5_med_alta_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Médico de alta con mayor cantidad de intervenciones: {max_hosp} intervenciones):")
    print(top5_med_alta_por_año[año].to_string(index=False))


Año 2019 (Médico de alta con mayor cantidad de intervenciones: 5165 intervenciones):
MEDICOALTA_ENCRIPTADO  count
               2932.0   5165
              10081.0   1211
                744.0    887
              16856.0    700
               8826.0    424

Año 2020 (Médico de alta con mayor cantidad de intervenciones: 2709 intervenciones):
MEDICOALTA_ENCRIPTADO  count
               2945.0   2709
               2932.0   1523
               5940.0    309
                744.0    243
              10081.0    223

Año 2021 (Médico de alta con mayor cantidad de intervenciones: 4340 intervenciones):
MEDICOALTA_ENCRIPTADO  count
             98438434   4340
             71851766    301
             92021088    224
             86916152    208
             68804909    169


In [17]:
for año in ["2022", "2023", "2024"]: # Médicos de alta más frecuentes entre 2022 y 2024
    max_hosp = top5_med_alta_por_año[año]['count'].iloc[0]
    print(f"\nAño {año} (Médico de alta con mayor cantidad de intervenciones: {max_hosp} intervenciones):")
    print(top5_med_alta_por_año[año].to_string(index=False))


Año 2022 (Médico de alta con mayor cantidad de intervenciones: 5519 intervenciones):
MEDICOALTA_ENCRIPTADO  count
             98438434   5519
             73663126    422
             71851766    301
             73199545    216
             74956461    182

Año 2023 (Médico de alta con mayor cantidad de intervenciones: 5098 intervenciones):
MEDICOALTA_ENCRIPTADO  count
             98438434   5098
             73663126    587
             71851766    393
             87138419    309
             73199545    303

Año 2024 (Médico de alta con mayor cantidad de intervenciones: 4924 intervenciones):
MEDICOALTA_ENCRIPTADO  count
             98438434   4924
             73663126    534
             77789698    364
             71851766    342
             74956461    248
